# UR5 equations of motion — three forward-dynamics pipelines

This notebook compares three ways to evaluate the UR5 joint accelerations $\ddot q$ from the standard manipulator equation

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau.
$$

All three paths target the **same** $\ddot q$ for a given $(q, \dot q, \tau)$. They differ in **how** $\ddot q$ is obtained and in **asymptotic cost** as the degree-of-freedom count $n$ grows.

| Pipeline | Idea | Catalog entry |
| --- | --- | --- |
| **RNEA–$H$** | Recursive Newton–Euler bias + explicit inertia solve | `UR5Manipulator.forward_dynamics` |
| **ABA** | Articulated-body algorithm (spatial $O(n)$) | `UR5Manipulator.forward_dynamics_aba` |
| **Symbolic Lagrange** | Derive $H,C,g$ once; evaluate numerically | `minilink.symbolic` → `to_minilink()` |

Companion script (same helpers): `run_demo.py` in this folder.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / "minilink").is_dir():
    repo = repo.parents[2]  # notebook in examples/projects/ur5_dynamics/
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

%matplotlib inline

## 1. Problem statement

For an $n$-link serial manipulator in generalized coordinates $q \in \mathbb{R}^n$, the second-order dynamics read

$$
H(q)\,\ddot q + b(q,\dot q) + d(q,\dot q) = \tau,
$$

where $H \succ 0$ is the joint-space inertia matrix, $b$ collects Coriolis, centrifugal, and gravity terms, $d$ is dissipation, and $\tau$ is the applied joint torque. **Forward dynamics** solves for $\ddot q$:

$$
\ddot q = H(q)^{-1}\bigl(\tau - b(q,\dot q) - d(q,\dot q)\bigr).
$$

The UR5 catalog model (`UR5Manipulator`) uses spatial-vector RNEA internally. For fair comparison below we **disable viscous damping** so the symbolic Lagrange export matches the spatial parameters.


## 2. Pipeline A — RNEA bias + explicit $H$ (RNEA–$H$)

### Mathematics

1. **Bias force** (inverse dynamics with zero acceleration):
   $$
   b(q,\dot q) = \mathrm{RNEA}(q,\dot q, 0).
   $$
   One outward/inward pass through the kinematic tree; includes gravity if enabled.

2. **Inertia matrix** column by column:
   $$
   H_{:,j} = \mathrm{RNEA}(q, 0, e_j), \qquad j = 1,\ldots,n,
   $$
   where $e_j$ is the $j$-th unit acceleration vector. Symmetrize $H \leftarrow \tfrac12(H + H^\top)$.

3. **Linear solve**:
   $$
   \ddot q = H^{-1}(\tau - b - d).
   $$

### Complexity (tree with $n$ joints)

| Step | Cost |
| --- | --- |
| Bias RNEA | $O(n)$ |
| Form $H$ ($n$ RNEA columns) | $O(n^2)$ |
| Solve $H\ddot q = \cdots$ | $O(n^3)$ — for UR5 ($n=6$) this is tiny |
| **Total per FD call** | **$O(n^2)$** dominated by forming $H$ |

**Strength:** exposes $H$, $g$, and (approximate) $C$ for analysis and control design. **Weakness:** forms the full $n \times n$ inertia at every step.


## 3. Pipeline B — Articulated Body Algorithm (ABA)

### Mathematics

ABA computes $\ddot q$ in **three passes** without ever assembling $H$:

1. **Outward pass:** link spatial velocities $v_i$, Coriolis accelerations $c_i$, and bias forces $p_{A,i} = \mathrm{crm}(v_i)^\top I_i v_i$.
2. **Inward pass:** articulated-body inertias $I_i^A$ and bias forces are propagated toward the base; joint scalars $U_i$, $d_i$, $u_i$ are accumulated.
3. **Second outward pass:** spatial accelerations are propagated and joint accelerations $\ddot q_i$ are extracted.

The result satisfies the same $\ddot q$ as the $H$-based formula when both use consistent spatial data.

### Complexity

| Step | Cost |
| --- | --- |
| All three passes | $O(n)$ |
| **Total per FD call** | **$O(n)$** |

**Strength:** optimal linear scaling for simulation of long chains. **Weakness:** does not produce $H$ directly (though extensions exist).


## 4. Pipeline C — Symbolic Lagrange derivation

### Mathematics

Build a DH chain in SymPy, define kinetic energy $T(q,\dot q)$ and potential $V(q)$, and apply the Euler–Lagrange equations:

$$
\frac{d}{dt}\frac{\partial T}{\partial \dot q} - \frac{\partial T}{\partial q} + \frac{\partial V}{\partial q} = \tau.
$$

This yields symbolic $H(q)$, $C(q,\dot q)$, and $g(q)$ which are **lambdified** once into a numeric `MechanicalSystem` via `to_minilink()`.

### Complexity

| Phase | Cost |
| --- | --- |
| Symbolic derivation (once) | Problem dependent; **minutes** for UR5 ($n=6$) |
| Export / lambdify (once) | Similar order to derivation |
| Each FD call (numeric $H$ + solve) | $O(n^2)$ evaluation + $O(n^3)$ solve |

**Strength:** exact automatic model generation; good for prototyping and verification. **Weakness:** upfront symbolic cost; runtime similar to RNEA–$H$ once exported.


## 5. Numerical study — UR5 case batch

We evaluate **67 configurations**: three named poses plus **64 random** samples with **fixed seed 0** for reproduction. Reference accelerations come from RNEA–$H$.


In [ ]:
import numpy as np

from examples.projects.ur5_dynamics.compare_eom import (
    DEFAULT_N_SAMPLES,
    DEFAULT_N_TIMING,
    DEFAULT_SEED,
    build_evaluation_batch,
    comprehensive_comparison,
    print_comprehensive_report,
    plot_comparison,
)
from examples.projects.ur5_dynamics.symbolic_ur5 import (
    build_symbolic_ur5,
    catalog_params_no_damping,
)
from minilink.dynamics.catalog.manipulators.ur5 import UR5Manipulator

SEED = DEFAULT_SEED
N_RANDOM = DEFAULT_N_SAMPLES
N_TIMING = DEFAULT_N_TIMING

arm = UR5Manipulator()
params = catalog_params_no_damping()
configs, labels = build_evaluation_batch(seed=SEED, n_random=N_RANDOM)
print(f"Batch: {len(labels)} samples ({len(labels) - N_RANDOM} named + {N_RANDOM} random, seed={SEED})")

In [ ]:
# First run derives and exports the symbolic UR5 (~2–3 min). Subsequent calls use a cache.
symbolic_plant = build_symbolic_ur5(verbose=True)

In [ ]:
result = comprehensive_comparison(
    arm,
    symbolic_plant,
    configs,
    params=params,
    seed=SEED,
    n_timing=N_TIMING,
    case_labels=labels,
)
print_comprehensive_report(result)

In [ ]:
fig, axes = plot_comparison(result)
fig.suptitle("UR5 forward dynamics — accuracy and timing", y=1.02)
fig

In [ ]:
import matplotlib.pyplot as plt

# Per-joint worst-case error across the full batch
joints = np.arange(1, 7)
fig2, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
ax.bar(joints - 0.15, result.methods["ABA"].per_joint_max, width=0.3, label="ABA")
ax.bar(joints + 0.15, result.methods["Symbolic"].per_joint_max, width=0.3, label="Symbolic")
ax.set_yscale("log")
ax.set_xlabel("Joint index")
ax.set_ylabel("max |Δqdd_j| vs RNEA–H")
ax.set_title("Worst per-joint error over all samples")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig2

## 6. Summary

On the UR5 with matched parameters and damping disabled:

* **ABA** and **symbolic Lagrange** agree with **RNEA–$H$** to near machine precision across the full random batch (max $|\Delta \ddot q|$ typically $\lesssim 10^{-10}$ rad/s$^2$).
* **ABA** is the fastest per forward-dynamics call for this $n=6$ chain; the speedup grows with $n$ because it avoids forming $H$.
* **Symbolic** derivation is a one-time cost; after export its runtime is comparable to RNEA–$H$ because both evaluate a dense $H$ and solve a linear system.

For simulation-heavy workflows on longer chains, prefer **ABA**. For teaching, linearization, or control design needing $H$ and $g$, keep **RNEA–$H$** or a pre-derived symbolic model.
